In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the CRUD class from the Project One module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Credentials for the aacuser account (hardcoded per project guidelines)
username = "aacuser"
password = "your_password_here"

# Connect to the database through the CRUD module
db = AnimalShelter(username, password)

# Rescue type queries from the Dashboard Specifications Document.
# Each age range uses a single key with both operators so neither bound
# overwrites the other.
RESCUE_QUERIES = {
    "water": {
        "animal_type": "Dog",
        "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever",
                          "Newfoundland"]},
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    },
    "mountain": {
        "animal_type": "Dog",
        "breed": {"$in": ["German Shepherd", "Alaskan Malamute",
                          "Old English Sheepdog", "Siberian Husky",
                          "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    },
    "disaster": {
        "animal_type": "Dog",
        "breed": {"$in": ["Doberman Pinscher", "German Shepherd",
                          "Golden Retriever", "Bloodhound", "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
    }
}


def get_animals(query):
    """Run a query through the CRUD module and return a clean dataframe."""
    dff = pd.DataFrame.from_records(db.read(query))
    # The ObjectId in '_id' cannot be displayed by the DataTable, so drop it
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff


# Starting state: every record in the collection (the "retrieve all" query)
df = get_animals({})

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare logo, encoded so it displays inside Jupyter
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Header: logo, title, and unique identifier
    html.Center([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 style={'height': '150px'}),
        html.H1('Grazioso Salvare Rescue Dog Dashboard'),
        html.H4('Developed by Leanne Byrne, CS-340')
    ]),
    html.Hr(),

    # Interactive filter: one radio button per rescue type, plus Reset
    html.Div([
        html.Label('Filter by rescue type:', style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True,
            labelStyle={'marginRight': '20px'}
        )
    ], style={'textAlign': 'center'}),
    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,               # 10 rows per page keeps the table readable
        sort_action='native',       # click a column header to sort
        filter_action='native',     # type in the box under a header to filter
        row_selectable='single',    # select one animal to show on the map
        selected_rows=[0],          # map shows the first row on load
        column_selectable='single',
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px'}
    ),
    html.Br(),
    html.Hr(),

    # Pie chart and geolocation map side by side
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(id='graph-id', className='col s12 m6',
                          style={'width': '50%'}),
                 html.Div(id='map-id', className='col s12 m6',
                          style={'width': '50%'})
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback([Output('datatable-id', 'data'),
               Output('datatable-id', 'selected_rows')],
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    """Requery MongoDB when the rescue type changes. Reset returns all records."""
    query = RESCUE_QUERIES.get(filter_type, {})
    dff = get_animals(query)
    # Select the first row of the new results so the map updates too
    selected = [0] if len(dff) > 0 else []
    return dff.to_dict('records'), selected


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    """Pie chart of breeds currently shown in the data table."""
    if not viewData:
        return [html.P("No animals match the current filter.")]

    dff = pd.DataFrame.from_dict(viewData)
    counts = dff['breed'].value_counts()

    # The unfiltered data has hundreds of breeds, so show the top 10
    # and group the rest as "Other" to keep the chart readable
    if len(counts) > 10:
        other_total = counts.iloc[10:].sum()
        counts = counts.head(10)
        counts['Other'] = other_total

    fig = px.pie(names=counts.index, values=counts.values,
                 title='Breeds in Current Results')
    return [dcc.Graph(figure=fig)]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')])
def update_styles(selected_columns):
    """Highlight a column when the user selects it."""
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]


@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    """Place a marker on the map for the selected animal."""
    if not viewData:
        return [html.P("No location to display.")]

    dff = pd.DataFrame.from_dict(viewData)

    # Single row selection, so use the first selected index (default to row 0)
    row = index[0] if index else 0
    if row >= len(dff):
        row = 0

    animal = dff.iloc[row]
    lat = animal['location_lat']
    lon = animal['location_long']

    # Some animals have no name on record
    name = animal.get('name')
    if not isinstance(name, str) or not name.strip():
        name = 'No name on record'

    return [
        dl.Map(style={'width': '100%', 'height': '500px'},
               center=[lat, lon], zoom=10, children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[lat, lon], children=[
                       dl.Tooltip(animal['breed']),
                       dl.Popup([
                           html.H1("Animal Name"),
                           html.P(name)
                       ])
                   ])
               ])
    ]


# Run the app. If port 8050 is in use from an earlier module, use app.run_server(port=8051)
app.run_server()

Dash app running on https://scorpiorebel-simplehostel-3000.codio.io/proxy/8050/
